# FinalGeo selector and protocol freeze — CPU only

This notebook performs the final pre-confirmatory selection. It enumerates all 91 endpoint-locked K=4 anchor sets, filters to 90–100% of Uniform anchor FLOPs, minimizes geometry radius, and writes the immutable seed-3/4/5 protocol. It does not train or load a checkpoint and does not read accuracy, exposure, gradient, prediction, or test data.

## Secure checkout
Create Kaggle secret `github_token`, attach `/kaggle/input/notebooks/dyhngg/test-rq2`, and select CPU/accelerator None.

In [ ]:
import os, subprocess, sys, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Selector revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Locate the frozen RQ2-v1 development evidence

In [ ]:
import importlib, json, pandas as pd
from IPython.display import display
import rq2_anchor_placement
import rq2_parameter_exposure
import rq2_finalgeo_selector
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_parameter_exposure = importlib.reload(rq2_parameter_exposure)
rq2_finalgeo_selector = importlib.reload(rq2_finalgeo_selector)
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT.exists(), f'Attach notebook output: {RQ2_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(
    RQ2_INPUT, '/kaggle/working/materialized-rq2-finalgeo'
)
print('Validated development root:', RQ2_ROOT)

## Select once and freeze
Rerunning with identical sources validates the same freeze. A changed source, rule, revision, or selected set is rejected once `finalgeo_frozen_protocol.json` exists.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/finalgeo-frozen')
started = time.perf_counter()
protocol = rq2_finalgeo_selector.freeze_finalgeo_selector(
    development_root=RQ2_ROOT,
    output_dir=OUTPUT_DIR,
    repo_root=PROJECT_ROOT,
)
print(f'Frozen in {time.perf_counter() - started:.2f} seconds')
print(json.dumps(protocol, indent=2))

## Inspect the deterministic selection

In [ ]:
candidates = pd.read_csv(OUTPUT_DIR / 'finalgeo_selector_candidates.csv')
display(candidates.loc[candidates['selected']])
display(candidates.loc[candidates['compute_feasible']].sort_values(
    'selection_rank_among_feasible'
).head(15))
assert protocol['status'] == 'FROZEN_BEFORE_CONFIRMATORY_SEEDS'
assert protocol['seeds_confirmatory'] == [3, 4, 5]
assert protocol['selection_rule']['objective'] == 'minimize_R_G'
assert protocol['selection_rule']['accuracy_used'] is False
assert protocol['selection_rule']['parameter_exposure_used'] is False

## Validate exactly three artifacts and export

In [ ]:
REQUIRED = [
    'finalgeo_selector_candidates.csv',
    'finalgeo_selected_anchors.json',
    'finalgeo_frozen_protocol.json',
]
actual = sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file())
assert actual == sorted(REQUIRED), f'Unexpected artifact set: {actual}'
assert len(candidates) == 91 and int(candidates['selected'].sum()) == 1
selected = candidates.loc[candidates['selected']].iloc[0]
assert 0.9 - 1e-12 <= selected['compute_ratio_vs_uniform'] <= 1.0 + 1e-12
bundle_path = Path('/kaggle/working/finalgeo-frozen-protocol.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for filename in REQUIRED:
        bundle.write(OUTPUT_DIR / filename, filename)
print('Persist this bundle before running seed 3:', bundle_path)
bundle_path